In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Load the dataset
print("Loading dataset...")
df = pd.read_csv('../data/raw/creditcard.csv')

# 2. Basic EDA
print(f"Dataset Shape: {df.shape}")
print("\nClass Distribution (0 = Normal, 1 = Fraud):")
print(df['Class'].value_counts(normalize=True) * 100)

# 3. Feature Engineering
print("\nEngineering new features...")

# Feature 1: Time_Hour (Convert elapsed seconds to hour of the day)
df['Time_Hour'] = (df['Time'] / 3600) % 24

# Feature 2: Amount_Log (Logarithmic transformation of Amount to handle extreme skewness)
# We use log1p (log(1+x)) to avoid errors with zero amounts
df['Amount_Log'] = np.log1p(df['Amount'])

# Feature 3: Is_High_Amount (Binary flag for transactions unusually high, e.g., > $200)
df['Is_High_Amount'] = (df['Amount'] > 200).astype(int)

# 4. Drop the original raw columns that we've transformed (optional but good practice)
df_processed = df.drop(columns=['Time', 'Amount'])

print(f"Processed Dataset Shape: {df_processed.shape}")
print("Feature Engineering Complete. New features added: 'Time_Hour', 'Amount_Log', 'Is_High_Amount'")

Loading dataset...
Dataset Shape: (284807, 31)

Class Distribution (0 = Normal, 1 = Fraud):
Class
0    99.827251
1     0.172749
Name: proportion, dtype: float64

Engineering new features...
Processed Dataset Shape: (284807, 32)
Feature Engineering Complete. New features added: 'Time_Hour', 'Amount_Log', 'Is_High_Amount'


In [2]:
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE
import os

# 1. Separate features (X) and target (y)
X = df_processed.drop('Class', axis=1)
y = df_processed['Class']

# 2. Split into training and testing sets (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print(f"Training set shape before SMOTE: {X_train.shape}")
print(f"Fraud cases in training set before SMOTE: {sum(y_train == 1)}")

# 3. Apply SMOTE only to the training data
smote = SMOTE(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)

print(f"Training set shape after SMOTE: {X_train_smote.shape}")
print(f"Fraud cases in training set after SMOTE: {sum(y_train_smote == 1)}")

# 4. Save the processed datasets
os.makedirs('../data/processed', exist_ok=True)

train_data = pd.concat([X_train_smote, y_train_smote], axis=1)
test_data = pd.concat([X_test, y_test], axis=1)

train_data.to_csv('../data/processed/train_smote.csv', index=False)
test_data.to_csv('../data/processed/test_data.csv', index=False)

print("Data processing complete. Files saved to data/processed/")

Training set shape before SMOTE: (227845, 31)
Fraud cases in training set before SMOTE: 394
Training set shape after SMOTE: (454902, 31)
Fraud cases in training set after SMOTE: 227451
Data processing complete. Files saved to data/processed/
